In [10]:
import torch
import os
import numpy as np
from chemflow.utils.utils import index_to_token, EdgeAligner

In [11]:
root_path = "/cluster/project/jorner/schmiste/flexflow/chemflow/outputs/2026-04-03/13-07-44"

data_root = "/cluster/project/jorner/schmiste/flexflow/chemflow/data/qm9/processed"
atom_token_path   = os.path.join(data_root, "atom_tokens.txt")
edge_token_path   = os.path.join(data_root, "edge_tokens.txt")
charge_token_path = os.path.join(data_root, "charge_tokens.txt")
distribution_path = os.path.join(data_root, "distributions.pt")

results = torch.load(os.path.join(root_path, "results.pt"), weights_only=False)

atom_tokens   = open(atom_token_path).read().splitlines()
edge_tokens   = open(edge_token_path).read().splitlines()
charge_tokens = open(charge_token_path).read().splitlines()

distributions = torch.load(distribution_path, weights_only=False)
coordinate_std = distributions["coordinate_std"]

In [12]:
def kabsch_align(P, Q):
    """Find optimal rotation R and translation t such that R @ P + t ≈ Q.

    Args:
        P: (K, 3) scaffold atom positions in the current frame.
        Q: (K, 3) scaffold atom positions in the reference frame (frame 0).

    Returns:
        R: (3, 3) rotation matrix
        t: (3,) translation vector
    """
    c_P = P.mean(axis=0)
    c_Q = Q.mean(axis=0)
    H = (P - c_P).T @ (Q - c_Q)
    U, _, Vt = np.linalg.svd(H)
    # Correct for reflection
    d = np.linalg.det(Vt.T @ U.T)
    R = Vt.T @ np.diag([1.0, 1.0, d]) @ U.T
    t = c_Q - R @ c_P
    return R, t


def align_trajectory_to_scaffold(traj_frames):
    """Apply Kabsch alignment to every frame so scaffold atoms stay at frame-0 positions.

    Uses `frame.scaffold_mask` (boolean tensor, True = OT-matched Real->Real core atom)
    as alignment anchors.  These atoms exist in every frame, so the anchor count K
    is constant across the whole trajectory.
    """
    ref_mask = traj_frames[0].scaffold_mask.cpu().numpy().astype(bool)
    ref_pos  = traj_frames[0].x.cpu().numpy()[ref_mask]   # (K, 3) reference

    aligned = []
    for frame in traj_frames:
        pos   = frame.x.cpu().numpy()                          # (N_t, 3)
        smask = frame.scaffold_mask.cpu().numpy().astype(bool) # (N_t,)
        R, t  = kabsch_align(pos[smask], ref_pos)

        aligned_pos = (R @ pos.T).T + t

        frame_aligned = frame.clone()
        frame_aligned.x = torch.from_numpy(aligned_pos.astype(np.float32))
        frame_aligned.scaffold_mask = frame.scaffold_mask.cpu()
        aligned.append(frame_aligned)

    return aligned

In [13]:
edge_aligner = EdgeAligner()


def process_traj(mol_list):
    new_trajs = []

    for mol in mol_list:
        x = mol.x
        x = x * coordinate_std
        a = mol.a.tolist()
        a = [index_to_token(atom_tokens, i) for i in a]
        a = [atom if atom != "<DEATH>" else "He" for atom in a]

        c = mol.c.tolist()
        c = [index_to_token(charge_tokens, i) for i in c]

        edge_infos = edge_aligner.align_edges(
            source_group=(mol.edge_index, [mol.e]),
        )
        edge_index_triu, e_pred_triu = (
            edge_infos["edge_index"],
            edge_infos["edge_attr"][0],
        )

        edge_index = edge_index_triu.T.tolist()
        e = e_pred_triu.tolist()
        e = [index_to_token(edge_tokens, i) for i in e]

        e_sanitized = []
        edge_index_sanitized = []
        for i, edge in enumerate(e):
            if edge == "<NO_BOND>":
                continue
            e_sanitized.append(int(edge))
            edge_index_sanitized.append(edge_index[i])

        scaffold_mask = mol.scaffold_mask.tolist() if hasattr(mol, "scaffold_mask") else None

        new_trajs.append(
            {
                "atoms": a,
                "pos": x,
                "edges": edge_index_sanitized,
                "edge_types": e_sanitized,
                "charges": c,
                "scaffold_mask": scaffold_mask,
            }
        )
    return new_trajs

In [14]:
import py3Dmol
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold


def _build_mol_block(frame):
    """Build an SDF mol block with true element names (preserves CPK colors)."""
    bond_mapping = {
        1: Chem.BondType.SINGLE,
        1.0: Chem.BondType.SINGLE,
        2: Chem.BondType.DOUBLE,
        2.0: Chem.BondType.DOUBLE,
        3: Chem.BondType.TRIPLE,
        1.5: Chem.BondType.AROMATIC,
    }
    mol = Chem.RWMol()
    for i, symbol in enumerate(frame["atoms"]):
        atom = Chem.Atom(symbol)
        atom.SetFormalCharge(int(frame["charges"][i]))
        mol.AddAtom(atom)
    for (src, dst), b_type in zip(frame["edges"], frame["edge_types"]):
        mol.AddBond(int(src), int(dst), bond_mapping.get(b_type, Chem.BondType.SINGLE))
    conf = Chem.Conformer(len(frame["atoms"]))
    for i, (x, y, z) in enumerate(frame["pos"]):
        conf.SetAtomPosition(i, (float(x), float(y), float(z)))
    mol.AddConformer(conf)
    return Chem.MolToMolBlock(mol)


def murcko_indices(frame):
    """Return sorted atom indices belonging to the Murcko scaffold of the frame's molecule."""
    mol = Chem.MolFromMolBlock(_build_mol_block(frame), removeHs=False)
    if mol is None:
        return []
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    return sorted(mol.GetSubstructMatch(scaffold))


def _apply_styles(view, orange_indices, red_indices):
    """Ball-and-stick for all atoms; transparent halo spheres for scaffold atoms."""
    view.setStyle({}, {"stick": {"radius": 0.15}, "sphere": {"scale": 0.25}})
    if orange_indices:
        view.addStyle(
            {"index": orange_indices},
            {"sphere": {"color": "orange", "scale": 0.4, "opacity": 0.2}},
        )
    if red_indices:
        view.addStyle(
            {"index": red_indices},
            {"sphere": {"color": "red", "scale": 0.4, "opacity": 0.2}},
        )


def visualize_variable_topology(trajectory_frames, orange_indices, red_indices, width=800, height=400):
    """Looping animation. Orange halos = sample Murcko scaffold, red = target Murcko scaffold."""
    combined_sdf = ""
    for frame in trajectory_frames:
        combined_sdf += _build_mol_block(frame) + "$$$$\n"
    view = py3Dmol.view(width=width, height=height)
    view.addModelsAsFrames(combined_sdf, "sdf")
    _apply_styles(view, orange_indices, red_indices)
    view.animate({"loop": "forward", "interval": 50})
    view.zoomTo()
    return view


def visualize_single_frame(frame, orange_indices=None, red_indices=None, width=400, height=400):
    """Render a single trajectory frame as a still interactive 3D view."""
    view = py3Dmol.view(width=width, height=height)
    view.addModel(_build_mol_block(frame), "sdf")
    _apply_styles(view, orange_indices or [], red_indices or [])
    view.zoomTo()
    return view

In [15]:
import ipywidgets as widgets
from IPython.display import display

# Pick a trajectory index to visualize
TRAJ_IDX = 0

# Align scaffold atoms to frame-0 reference
aligned_traj = align_trajectory_to_scaffold(results[TRAJ_IDX])

# Decode to atoms/bonds/coords
frames = process_traj(aligned_traj)

# ── Murcko scaffold indices ───────────────────────────────────────────────────
orange_idx = None #murcko_indices(frames[0])
red_idx    = None #murcko_indices(frames[-1])



def _print_murcko(indices, own_frame, other_frame, label, other_label):
    print(f"{label} — Murcko scaffold ({len(indices)} atoms):")
    print(f"  {'idx':>4}  {'own':>4}  {other_label}")
    for i in indices:
        own   = own_frame["atoms"][i]
        other = other_frame["atoms"][i] if i < len(other_frame["atoms"]) else "n/a"
        print(f"  {i:>4}  {own:>4}  {other}")

#_print_murcko(orange_idx, frames[0],  frames[-1], "Sample (frame 0)  [orange]", f"@ target (frame {len(frames)-1})")
#print()
#_print_murcko(red_idx,    frames[-1], frames[0],  f"Target (frame {len(frames)-1})   [red]", "@ sample (frame 0)")
#print()

# ── Animation (auto-loops) ────────────────────────────────────────────────────
print("Auto-playing animation:")
visualize_variable_topology(frames, orange_idx, red_idx).show()

# ── Still frames: first and last ──────────────────────────────────────────────
print("Frame 0  (start)")
visualize_single_frame(frames[0],  orange_indices=orange_idx, width=400, height=350).show()

print(f"Frame {len(frames)-1}  (end)")
visualize_single_frame(frames[-1], red_indices=red_idx, width=400, height=350).show()

Auto-playing animation:


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Frame 0  (start)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Frame 100  (end)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [16]:
# ── Frame slider (self-contained HTML — no Python callbacks) ──────────────────
# The slider calls the 3Dmol viewer's JS API directly in the browser.
# Relies on $3Dmolpromise already being defined by the animation cell above.
from IPython.display import HTML, display as _display

# Build SDF with all frames (no animate — slider controls the frame)
_combined_sdf = "".join(_build_mol_block(f) + "$$$$\n" for f in frames)

_v = py3Dmol.view(width=700, height=420)
_v.addModelsAsFrames(_combined_sdf, "sdf")
_apply_styles(_v, orange_idx, red_idx)
_v.setFrame(0)
_v.zoomTo()

_html = _v.write_html()
_uid  = _v.uniqueid

# Make viewer var global so the slider's oninput can reach it
# (py3Dmol uses `var viewer_UID` inside a .then() closure — not window-scoped)
_html = _html.replace(f"var viewer_{_uid} =", f"window.viewer_{_uid} =")

_slider = f"""
<div style="margin:6px 0">
  <input type="range" min="0" max="{len(frames)-1}" value="0" style="width:600px"
         oninput="this.nextElementSibling.textContent = 'Frame ' + this.value;
                  window.viewer_{_uid}.setFrame(parseInt(this.value));
                  window.viewer_{_uid}.render();">
  <span style="margin-left:8px">Frame 0</span>
</div>
"""

_display(HTML(_slider + _html))